# 第五章：级联评估

**系列：** OpenEvolve — 从零到专家·手撕全流程  
**上一章：** [04 — LLM 作为变异算子](./04-llm-mutator.ipynb)  
**本章内容：** 为什么需要多阶段评估、级联评估的原理与实现、阈值机制、超时处理、以及 Artifact（工件）系统。

---

## 1. 痛点：评估太慢了！

上一章我们用 LLM 生成了智能变异，但每次评估都跑**完整的测试套件**：

```
进化循环第 1 轮：
  生成变异代码 ... 0.5 秒
  完整评估     ... 30 秒  ← 大部分时间花在这里！
  结果：代码根本跑不通（语法错误）

进化循环第 2 轮：
  生成变异代码 ... 0.5 秒
  完整评估     ... 30 秒  ← 又白等了！
  结果：跑通了但正确性只有 20%
```

问题很明显：**大量时间浪费在注定失败的程序上**。

想象一个招聘流程：如果每个候选人都做完全部面试（笔试 + 技术面 + 终面 + 背调），那效率极低。
实际上，我们会先**简历筛选** → 再**笔试** → 最后才**面试**——逐步增加投入。

这就是**级联评估（Cascade Evaluation）**的核心思想。

## 2. 理论：级联评估模型

### 2.1 三阶段管道

级联评估把一次评估拆成多个**阶段**，每个阶段有一个**通过阈值**：

```mermaid
flowchart LR
    P[待评估程序] --> S1["阶段1：基础验证<br/>阈值=0.5"]
    S1 -->|通过| S2["阶段2：中等测试<br/>阈值=0.75"]
    S1 -->|未通过| F1[❌ 淘汰]
    S2 -->|通过| S3["阶段3：全面评估<br/>阈值=0.9"]
    S2 -->|未通过| F2[❌ 淘汰]
    S3 -->|完成| R[✅ 最终结果]
```

### 2.2 核心公式

**阈值判定函数**：

$$\text{pass}(M, \theta) = \begin{cases} \text{True} & \text{if } \bar{M} \geq \theta \\ \text{False} & \text{otherwise} \end{cases}$$

其中 $\bar{M}$ 是所有数值指标的平均分，$\theta$ 是当前阶段的阈值。

**通俗解释：** 就像考试——你所有科目的平均分必须达到及格线才能进入下一轮。

**数值例子：**
> 排序程序在阶段 1 返回两个指标：runs_successfully=1.0, basic_correctness=0.6  
> $\bar{M} = \frac{1.0 + 0.6}{2} = 0.8$  
> 阶段 1 阈值 $\theta_1 = 0.5$  
> 因为 $0.8 \geq 0.5$，通过！进入阶段 2。  

**特殊规则：** 如果指标中包含 `combined_score`，则直接使用该值（不取平均）：

$$\bar{M} = \begin{cases} M_{\text{combined\_score}} & \text{if combined\_score exists} \\ \frac{1}{K}\sum_{k=1}^{K} m_k & \text{otherwise} \end{cases}$$

**生活类比：** 高考有总分和单科分——如果报了「总分」，按总分判断；如果没报总分，就按各科平均分判断。

### 2.3 时间节约估算

假设 Stage 1 耗时 1 秒，Stage 2 耗时 5 秒，Stage 3 耗时 30 秒。  
如果 70% 的程序在 Stage 1 就被淘汰，15% 在 Stage 2 被淘汰，只有 15% 进入 Stage 3：

$$\text{平均耗时} = 0.70 \times 1 + 0.15 \times (1+5) + 0.15 \times (1+5+30) = 0.7 + 0.9 + 5.4 = 7.0 \text{ 秒}$$

**对比：** 如果全部跑完整评估，平均耗时 = 36 秒。级联评估节约了 **80%** 的时间！

## 3. 从零实现：评估结果结构

在写评估器之前，先定义评估结果的数据结构。一个评估结果包含两部分：
- **metrics（指标）**：数值型得分，用于计算适应度和阈值判定
- **artifacts（工件）**：调试信息、日志等非数值数据

| 概念 | 类比 | 例子 |
|------|------|------|
| metrics | 考试分数 | `{"correctness": 0.9, "speed": 0.7}` |
| artifacts | 考试草稿纸 | `{"stderr": "warning: ...", "timing_details": {...}}` |

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Union


@dataclass
class EvaluationResult:
    """评估结果：指标 + 工件"""
    metrics: Dict[str, float]  # 数值得分
    artifacts: Dict[str, Union[str, bytes]] = field(default_factory=dict)  # 调试数据
    
    def has_artifacts(self) -> bool:
        return len(self.artifacts) > 0
    
    def get_total_artifact_size(self) -> int:
        """计算所有工件的总字节数"""
        total = 0
        for v in self.artifacts.values():
            if isinstance(v, bytes):
                total += len(v)
            elif isinstance(v, str):
                total += len(v.encode('utf-8'))
            else:
                total += len(str(v).encode('utf-8'))
        return total


# 验证
r = EvaluationResult(
    metrics={"correctness": 0.9, "speed": 0.7},
    artifacts={"stderr": "all good", "timing": "0.3s"}
)
print(f"指标: {r.metrics}")
print(f"有工件? {r.has_artifacts()}")
print(f"工件总大小: {r.get_total_artifact_size()} bytes")

## 4. 阈值判定函数

级联评估的核心：判定一个程序是否**达标**。

OpenEvolve 的判定逻辑有两条路径：
1. 如果存在 `combined_score` 指标 → 直接用它
2. 否则 → 计算所有数值指标的平均值（排除 `error` 字段）

In [ ]:
def passes_threshold(metrics: dict, threshold: float) -> bool:
    """判断指标是否达到阈值
    
    优先使用 combined_score，否则取所有数值指标的平均值。
    """
    # 路径 1：有 combined_score 就直接用
    if 'combined_score' in metrics:
        return float(metrics['combined_score']) >= threshold
    
    # 路径 2：取所有数值指标的平均（排除 error 字段）
    valid = [
        float(v) for k, v in metrics.items()
        if k != 'error' and isinstance(v, (int, float))
    ]
    
    if not valid:
        return False
    
    avg = sum(valid) / len(valid)
    return avg >= threshold


# 数值例子演示
print("=== 阈值判定演示 ===")

# 例 1：有 combined_score
m1 = {"combined_score": 0.85, "accuracy": 0.9, "speed": 0.3}
print(f"例1 (combined_score=0.85, 阈值=0.75): {passes_threshold(m1, 0.75)}")
# 直接看 combined_score=0.85 >= 0.75 → True

# 例 2：没有 combined_score，取平均
m2 = {"accuracy": 0.9, "speed": 0.4}
avg = (0.9 + 0.4) / 2  # = 0.65
print(f"例2 (accuracy=0.9, speed=0.4, 平均={avg}, 阈值=0.75): {passes_threshold(m2, 0.75)}")
# 平均 0.65 < 0.75 → False

# 例 3：error 字段被排除
m3 = {"accuracy": 0.9, "speed": 0.8, "error": 0.0}
avg3 = (0.9 + 0.8) / 2  # error 被排除
print(f"例3 (error 被排除, 平均={avg3}, 阈值=0.75): {passes_threshold(m3, 0.75)}")

## 5. 从零实现：级联评估器

现在把所有零件组装起来。级联评估器的职责：
1. 按顺序运行 Stage 1 → Stage 2 → Stage 3
2. 每个阶段判定阈值，不通过则**提前终止**
3. 合并各阶段的指标和工件
4. 处理**超时**（程序跑太久就杀掉）

### 设计选择

| 问题 | OpenEvolve 的选择 | 原因 |
|------|-------------------|------|
| 阈值判定 | 平均值 >= 阈值 | 简单直觉、适合多指标 |
| 阶段数 | 3 阶段 | 经验平衡点（太少不够分，太多管理复杂） |
| 超时处理 | 整个评估一个超时 | 避免参数爆炸 |
| 指标合并 | 后阶段覆盖同名指标 | 后阶段更全面 |

In [ ]:
import subprocess
import tempfile
import os
import time
from typing import List, Optional, Callable


class CascadeEvaluator:
    """级联评估器：多阶段渐进式评估"""
    
    def __init__(
        self,
        stages: List[Callable],  # [stage1_fn, stage2_fn, stage3_fn]
        thresholds: List[float] = None,  # 每阶段的通过阈值
        timeout: float = 30.0,  # 总超时（秒）
    ):
        self.stages = stages
        self.thresholds = thresholds or [0.5, 0.75, 0.9]
        self.timeout = timeout
        
        # 验证：阶段数和阈值数必须匹配
        assert len(self.stages) == len(self.thresholds), (
            f"阶段数({len(self.stages)}) != 阈值数({len(self.thresholds)})"
        )
    
    def evaluate(self, program_code: str) -> EvaluationResult:
        """级联评估一个程序
        
        返回合并后的 EvaluationResult：
        - metrics 包含所有通过阶段的指标
        - artifacts 包含调试信息和失败阶段信息
        """
        merged_metrics = {}
        merged_artifacts = {}
        start_time = time.time()
        
        for i, (stage_fn, threshold) in enumerate(zip(self.stages, self.thresholds)):
            stage_name = f"stage{i+1}"
            elapsed = time.time() - start_time
            
            # 超时检查
            if elapsed >= self.timeout:
                merged_metrics[f"{stage_name}_passed"] = 0.0
                merged_artifacts["failure_reason"] = "timeout"
                merged_artifacts["timeout_at_stage"] = stage_name
                break
            
            try:
                # 运行当前阶段
                result = stage_fn(program_code)
                
                # 结果可以是 dict 或 EvaluationResult
                if isinstance(result, EvaluationResult):
                    stage_metrics = result.metrics
                    stage_artifacts = result.artifacts
                elif isinstance(result, dict):
                    stage_metrics = result
                    stage_artifacts = {}
                else:
                    stage_metrics = {}
                    stage_artifacts = {"error": f"Unexpected result type: {type(result)}"}
                
                # 合并指标（后阶段可覆盖前阶段同名指标）
                merged_metrics.update(stage_metrics)
                merged_artifacts.update(stage_artifacts)
                
                # 阈值判定
                if not passes_threshold(stage_metrics, threshold):
                    merged_metrics[f"{stage_name}_passed"] = 0.0
                    merged_artifacts["failure_stage"] = stage_name
                    merged_artifacts["failure_threshold"] = threshold
                    break
                else:
                    merged_metrics[f"{stage_name}_passed"] = 1.0
                    
            except Exception as e:
                # 捕获评估中的异常
                merged_metrics[f"{stage_name}_passed"] = 0.0
                merged_metrics["error"] = 0.0
                merged_artifacts["failure_stage"] = stage_name
                merged_artifacts["exception"] = str(e)
                break
        
        return EvaluationResult(metrics=merged_metrics, artifacts=merged_artifacts)


print("CascadeEvaluator 定义完成 ✓")

## 6. 排序程序的三阶段评估

回到我们的贯穿例子——排序算法。定义三个评估阶段：

| 阶段 | 做什么 | 阈值 | 类比 |
|------|--------|------|------|
| Stage 1 | 代码能跑吗？ | 0.5 | 简历筛选：你是程序员吗？ |
| Stage 2 | 排序结果正确吗？ | 0.75 | 笔试：基本功怎么样？ |
| Stage 3 | 性能和边界情况 | 0.9 | 终面：综合实力如何？ |

In [ ]:
import random
import time as time_module


def sort_stage1(program_code: str) -> dict:
    """阶段1：基础验证——代码能运行吗？
    
    只用一个极小的输入检查：
    - 代码是否有语法错误？
    - 函数是否存在？
    - 是否能返回一个列表？
    """
    try:
        namespace = {}
        exec(program_code, namespace)
        
        if 'sort_func' not in namespace:
            return {"runs_successfully": 0.0, "has_function": 0.0}
        
        result = namespace['sort_func']([3, 1, 2])
        is_list = isinstance(result, list)
        
        return {
            "runs_successfully": 1.0,
            "has_function": 1.0,
            "returns_list": 1.0 if is_list else 0.0
        }
    except Exception as e:
        return {"runs_successfully": 0.0, "error_msg": 0.0}


def sort_stage2(program_code: str) -> dict:
    """阶段2：正确性测试——排序结果对吗？
    
    用 5 个测试用例验证：
    - 普通数组、空数组、已排序、逆序、重复元素
    """
    test_cases = [
        ([5, 3, 8, 1, 9], [1, 3, 5, 8, 9]),
        ([], []),
        ([1, 2, 3], [1, 2, 3]),
        ([3, 2, 1], [1, 2, 3]),
        ([2, 2, 1, 1], [1, 1, 2, 2]),
    ]
    
    namespace = {}
    exec(program_code, namespace)
    sort_func = namespace['sort_func']
    
    passed = 0
    for inp, expected in test_cases:
        try:
            if sort_func(inp[:]) == expected:
                passed += 1
        except Exception:
            pass
    
    correctness = passed / len(test_cases)
    return {"correctness": correctness, "test_cases_passed": passed / len(test_cases)}


def sort_stage3(program_code: str) -> EvaluationResult:
    """阶段3：全面评估——性能 + 边界 + 稳定性
    
    注意：返回 EvaluationResult 而不是 dict，展示两种返回格式。
    """
    namespace = {}
    exec(program_code, namespace)
    sort_func = namespace['sort_func']
    
    # 性能测试：排序 1000 个元素
    big_list = list(range(1000, 0, -1))
    start = time_module.perf_counter()
    result = sort_func(big_list[:])
    elapsed = time_module.perf_counter() - start
    
    # 性能得分：1 秒内满分，越快越高
    speed_score = max(0.0, 1.0 - elapsed)
    
    # 大数组正确性
    big_correct = 1.0 if result == sorted(big_list) else 0.0
    
    # 边界测试：单元素
    single = 1.0 if sort_func([42]) == [42] else 0.0
    
    # 负数测试
    neg = 1.0 if sort_func([-3, -1, -2]) == [-3, -2, -1] else 0.0
    
    combined = (big_correct * 0.4 + speed_score * 0.2 + single * 0.2 + neg * 0.2)
    
    return EvaluationResult(
        metrics={
            "combined_score": combined,
            "speed_score": speed_score,
            "big_array_correct": big_correct,
            "single_element": single,
            "negative_numbers": neg,
        },
        artifacts={
            "execution_time": f"{elapsed:.4f}s",
            "array_size_tested": "1000",
        }
    )


print("三个评估阶段定义完成 ✓")
print(f"  Stage 1: 基础验证 (阈值 0.5)")
print(f"  Stage 2: 正确性测试 (阈值 0.75)")
print(f"  Stage 3: 全面评估 (阈值 0.9)")

## 7. 运行级联评估演示

用三种不同质量的排序程序测试级联评估器：
1. **语法错误** → 预期在 Stage 1 被淘汰
2. **部分正确** → 预期在 Stage 2 被淘汰
3. **高质量** → 预期通过所有阶段

In [ ]:
# 创建级联评估器
evaluator = CascadeEvaluator(
    stages=[sort_stage1, sort_stage2, sort_stage3],
    thresholds=[0.5, 0.75, 0.9],
    timeout=10.0
)

# === 程序 A：语法错误 ===
bad_code = "def sort_func(arr)\n    return arr"  # 缺少冒号
result_a = evaluator.evaluate(bad_code)
print("=== 程序 A (语法错误) ===")
print(f"  指标: {result_a.metrics}")
print(f"  失败阶段: {result_a.artifacts.get('failure_stage', 'N/A')}")
print()

# === 程序 B：部分正确（只对非空数组有效，空数组返回 None） ===
partial_code = '''
def sort_func(arr):
    if not arr:
        return None  # Bug: 空数组应返回 []
    for i in range(len(arr)):
        for j in range(i+1, len(arr)):
            if arr[i] > arr[j]:
                arr[i], arr[j] = arr[j], arr[i]
    return arr
'''
result_b = evaluator.evaluate(partial_code)
print("=== 程序 B (部分正确) ===")
print(f"  指标: {result_b.metrics}")
print(f"  失败阶段: {result_b.artifacts.get('failure_stage', 'N/A')}")
print()

# === 程序 C：高质量冒泡排序 ===
good_code = '''
def sort_func(arr):
    arr = arr[:]  # 不修改原数组
    n = len(arr)
    for i in range(n):
        for j in range(0, n-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr
'''
result_c = evaluator.evaluate(good_code)
print("=== 程序 C (高质量) ===")
print(f"  指标: {result_c.metrics}")
print(f"  工件: {result_c.artifacts}")
print(f"  通过所有阶段? {result_c.metrics.get('stage3_passed', 0) == 1.0}")

## 8. 可视化：级联评估的过滤效果

让我们模拟一批程序通过级联评估，看看每个阶段淘汰了多少。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# 模拟 50 个不同质量的排序程序
programs = []

# 30% 语法错误
for _ in range(15):
    programs.append("def sort_func(arr\n  return arr")

# 30% 部分正确
for _ in range(15):
    programs.append('''
def sort_func(arr):
    if not arr:
        return None
    for i in range(len(arr)):
        for j in range(i+1, len(arr)):
            if arr[i] > arr[j]:
                arr[i], arr[j] = arr[j], arr[i]
    return arr
''')

# 20% 正确但慢
for _ in range(10):
    programs.append('''
def sort_func(arr):
    arr = arr[:]
    for i in range(len(arr)):
        for j in range(0, len(arr)-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr
''')

# 20% 高质量（快速排序）
for _ in range(10):
    programs.append('''
def sort_func(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr)//2]
    left = [x for x in arr if x < pivot]
    mid = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return sort_func(left) + mid + sort_func(right)
''')

# 评估所有程序，追踪每阶段的存活数
stage_counts = {"提交": len(programs), "Stage 1 通过": 0, "Stage 2 通过": 0, "Stage 3 通过": 0}
stage_times = []  # 每个程序的评估耗时

for prog in programs:
    start = time_module.perf_counter()
    result = evaluator.evaluate(prog)
    elapsed = time_module.perf_counter() - start
    stage_times.append(elapsed)
    
    if result.metrics.get("stage1_passed", 0) == 1.0:
        stage_counts["Stage 1 通过"] += 1
    if result.metrics.get("stage2_passed", 0) == 1.0:
        stage_counts["Stage 2 通过"] += 1
    if result.metrics.get("stage3_passed", 0) == 1.0:
        stage_counts["Stage 3 通过"] += 1

# 绘制漏斗图
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：漏斗
labels = list(stage_counts.keys())
counts = list(stage_counts.values())
colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
bars = ax1.barh(labels[::-1], counts[::-1], color=colors[::-1])
for bar, count in zip(bars, counts[::-1]):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{count} ({count/len(programs)*100:.0f}%)',
             va='center', fontsize=12)
ax1.set_xlabel('programs', fontsize=12)
ax1.set_title('Cascade Evaluation Funnel', fontsize=14)
ax1.set_xlim(0, max(counts) * 1.3)

# 右图：时间节约
full_eval_time = max(stage_times) * len(programs)  # 假设全部跑完整评估
cascade_time = sum(stage_times)
ax2.bar(['Full Eval', 'Cascade Eval'], 
        [full_eval_time, cascade_time],
        color=['#e74c3c', '#2ecc71'])
ax2.set_ylabel('Total Time (s)', fontsize=12)
ax2.set_title('Time Comparison', fontsize=14)
savings = (1 - cascade_time / full_eval_time) * 100 if full_eval_time > 0 else 0
ax2.text(1, cascade_time + 0.01, f'Saved {savings:.0f}%', 
         ha='center', fontsize=14, fontweight='bold', color='green')

plt.tight_layout()
plt.show()

print(f"\n统计：")
for label, count in stage_counts.items():
    print(f"  {label}: {count} / {len(programs)}")

## 9. 集成到进化循环

现在把级联评估器接入前几章构建的进化循环中。

关键改动：
- 之前：`score = evaluate(code)` （单一评估）
- 现在：`result = cascade_evaluator.evaluate(code)` → `score = avg(result.metrics)`
- 新增：失败程序的工件（artifacts）可以**反馈给 LLM**，让它知道为什么失败

In [ ]:
def evolution_with_cascade(
    initial_code: str,
    evaluator: CascadeEvaluator,
    mutate_fn,
    generations: int = 20,
):
    """带级联评估的进化循环"""
    best_code = initial_code
    best_result = evaluator.evaluate(initial_code)
    best_score = _compute_fitness(best_result.metrics)
    
    history = {
        "scores": [best_score],
        "stages_reached": [],  # 每代到达的最高阶段
        "evaluations_saved": 0,  # 级联节省的评估数
    }
    
    for gen in range(generations):
        # 变异
        child_code = mutate_fn(best_code)
        
        # 级联评估
        child_result = evaluator.evaluate(child_code)
        child_score = _compute_fitness(child_result.metrics)
        
        # 追踪到达了哪个阶段
        max_stage = 0
        for i in range(1, 4):
            if child_result.metrics.get(f"stage{i}_passed", 0) == 1.0:
                max_stage = i
        history["stages_reached"].append(max_stage)
        
        # 如果没到 Stage 3，说明级联帮我们省了时间
        if max_stage < 3:
            history["evaluations_saved"] += 1
        
        # (1+1)-ES 选择
        if child_score > best_score:
            best_code = child_code
            best_score = child_score
            best_result = child_result
        
        history["scores"].append(best_score)
    
    return best_code, best_result, history


def _compute_fitness(metrics: dict) -> float:
    """从指标计算适应度（与阈值判定类似）"""
    if 'combined_score' in metrics:
        return float(metrics['combined_score'])
    valid = [float(v) for k, v in metrics.items()
             if k != 'error' and isinstance(v, (int, float))]
    return sum(valid) / len(valid) if valid else 0.0


# 简单随机变异（复用第一章的思路）
def random_sort_mutator(code: str) -> str:
    """随机生成一种排序实现"""
    variants = [
        # 冒泡排序（正确）
        '''def sort_func(arr):
    arr = arr[:]
    for i in range(len(arr)):
        for j in range(0, len(arr)-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr''',
        # 选择排序（正确）
        '''def sort_func(arr):
    arr = arr[:]
    for i in range(len(arr)):
        min_idx = i
        for j in range(i+1, len(arr)):
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]
    return arr''',
        # 有 bug 的版本
        '''def sort_func(arr):
    return arr  # Bug: 直接返回''',
        # 语法错误版本
        '''def sort_func(arr
    return sorted(arr)''',
        # 快速排序（正确，高性能）
        '''def sort_func(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr)//2]
    left = [x for x in arr if x < pivot]
    mid = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return sort_func(left) + mid + sort_func(right)''',
    ]
    return random.choice(variants)


# 运行带级联评估的进化
random.seed(42)
initial = '''def sort_func(arr):
    arr = arr[:]
    for i in range(len(arr)):
        for j in range(0, len(arr)-i-1):
            if arr[j] > arr[j+1]:
                arr[j], arr[j+1] = arr[j+1], arr[j]
    return arr'''

best_code, best_result, history = evolution_with_cascade(
    initial_code=initial,
    evaluator=evaluator,
    mutate_fn=random_sort_mutator,
    generations=30,
)

print(f"最佳适应度: {_compute_fitness(best_result.metrics):.3f}")
print(f"级联节省的评估: {history['evaluations_saved']}/{30}")
print(f"到达各阶段的分布:")
for stage in range(4):
    count = history['stages_reached'].count(stage)
    print(f"  Stage {stage}: {count} 次 ({count/30*100:.0f}%)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：适应度曲线
ax1.plot(history['scores'], 'b-', linewidth=2)
ax1.set_xlabel('Generation', fontsize=12)
ax1.set_ylabel('Fitness', fontsize=12)
ax1.set_title('Fitness over Generations (with Cascade)', fontsize=14)
ax1.grid(True, alpha=0.3)

# 右图：每代到达的阶段
stages = history['stages_reached']
colors_map = {0: '#e74c3c', 1: '#f39c12', 2: '#3498db', 3: '#2ecc71'}
stage_colors = [colors_map[s] for s in stages]
ax2.bar(range(len(stages)), [s+0.5 for s in stages], color=stage_colors, alpha=0.8)
ax2.set_xlabel('Generation', fontsize=12)
ax2.set_ylabel('Stage Reached', fontsize=12)
ax2.set_title('Highest Stage Reached per Generation', fontsize=14)
ax2.set_yticks([0.5, 1.5, 2.5, 3.5])
ax2.set_yticklabels(['Failed', 'Stage 1', 'Stage 2', 'Stage 3'])

# 图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#e74c3c', label='Failed (Stage 0)'),
    Patch(facecolor='#f39c12', label='Stage 1 only'),
    Patch(facecolor='#3498db', label='Stage 2'),
    Patch(facecolor='#2ecc71', label='Stage 3 (full)'),
]
ax2.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.show()

## 10. 进阶：工件反馈循环

级联评估不只是省时间——**失败信息本身就是宝贵的数据**。

当一个程序在某阶段失败时，评估器收集的 artifacts 可以告诉 LLM **哪里出了问题**：

```
传统方式：
  LLM: 请改进这段代码 → 新代码 → 评估失败 → LLM: 请改进这段代码（盲目重试）

有 artifacts 的方式：
  LLM: 请改进这段代码 → 新代码 → Stage 2 失败
  → artifacts: {failure_stage: 'stage2', correctness: 0.6, test_cases_passed: 3/5}
  → LLM: 代码在正确性测试中失败了，只通过了 3/5 个测试 → 更有针对性的修改
```

这在第四章的 PromptBuilder 中可以直接使用——把失败工件加入 prompt 的历史记录中。

In [ ]:
def format_failure_feedback(result: EvaluationResult) -> str:
    """将评估失败的信息格式化为可供 LLM 阅读的反馈"""
    lines = []
    
    failure_stage = result.artifacts.get('failure_stage', 'unknown')
    lines.append(f"## Evaluation Failed at {failure_stage}")
    lines.append("")
    
    # 列出所有指标
    lines.append("### Metrics:")
    for k, v in sorted(result.metrics.items()):
        if isinstance(v, float):
            status = 'PASS' if v >= 0.5 else 'FAIL'
            lines.append(f"  - {k}: {v:.2f} [{status}]")
    
    # 列出有用的工件
    if result.artifacts:
        lines.append("")
        lines.append("### Debug Info:")
        for k, v in result.artifacts.items():
            if k not in ('failure_stage', 'failure_threshold'):
                lines.append(f"  - {k}: {v}")
    
    return '\n'.join(lines)


# 演示：用部分正确的程序
partial = '''
def sort_func(arr):
    if not arr:
        return None
    for i in range(len(arr)):
        for j in range(i+1, len(arr)):
            if arr[i] > arr[j]:
                arr[i], arr[j] = arr[j], arr[i]
    return arr
'''
result = evaluator.evaluate(partial)
feedback = format_failure_feedback(result)
print("=== 失败反馈（可直接加入 LLM Prompt）===")
print(feedback)

## 11. 工件存储策略

OpenEvolve 中，工件的存储有两种模式：

| 大小 | 存储方式 | 原因 |
|------|---------|------|
| < 32KB | 内存（JSON 字典） | 快速访问，适合小数据 |
| >= 32KB | 磁盘文件 | 避免内存爆炸 |

**数值例子：**
> 一个程序的工件有：`stderr`（200 bytes）+ `timing_details`（500 bytes）= 700 bytes < 32KB → 存内存  
> 另一个程序的工件有：`full_output_log`（50KB）→ 存磁盘

**生活类比：** 就像收据管理——小额收据夹在钱包里（内存），大额发票存在文件柜里（磁盘）。

In [ ]:
import json

ARTIFACT_SIZE_THRESHOLD = 32 * 1024  # 32KB


class ArtifactStore:
    """工件存储器：小的存内存，大的存磁盘"""
    
    def __init__(self, disk_dir: str = "/tmp/artifacts"):
        self.memory_store = {}  # program_id -> artifacts_dict
        self.disk_dir = disk_dir
        os.makedirs(disk_dir, exist_ok=True)
    
    def store(self, program_id: str, result: EvaluationResult):
        """根据大小决定存储位置"""
        if not result.has_artifacts():
            return
        
        total_size = result.get_total_artifact_size()
        
        if total_size < ARTIFACT_SIZE_THRESHOLD:
            # 小工件 → 内存
            self.memory_store[program_id] = result.artifacts
            return "memory"
        else:
            # 大工件 → 磁盘
            path = os.path.join(self.disk_dir, f"{program_id}.json")
            # 转换 bytes 为 base64 字符串
            serializable = {}
            for k, v in result.artifacts.items():
                if isinstance(v, bytes):
                    import base64
                    serializable[k] = {"_type": "bytes", "data": base64.b64encode(v).decode()}
                else:
                    serializable[k] = v
            with open(path, 'w') as f:
                json.dump(serializable, f)
            return "disk"
    
    def retrieve(self, program_id: str) -> dict:
        """获取工件（自动判断来源）"""
        if program_id in self.memory_store:
            return self.memory_store[program_id]
        
        path = os.path.join(self.disk_dir, f"{program_id}.json")
        if os.path.exists(path):
            with open(path, 'r') as f:
                return json.load(f)
        
        return {}


# 演示
store = ArtifactStore()

# 小工件
small_result = EvaluationResult(
    metrics={"score": 0.8},
    artifacts={"stderr": "OK", "time": "0.5s"}
)
location = store.store("prog_001", small_result)
print(f"小工件 ({small_result.get_total_artifact_size()} bytes) → {location}")

# 大工件
big_result = EvaluationResult(
    metrics={"score": 0.9},
    artifacts={"full_log": "x" * 50000}  # 50KB
)
location = store.store("prog_002", big_result)
print(f"大工件 ({big_result.get_total_artifact_size()} bytes) → {location}")

# 检索
print(f"\n检索 prog_001: {store.retrieve('prog_001')}")
print(f"检索 prog_002: 大小={len(json.dumps(store.retrieve('prog_002')))} chars")

## 12. 我们的实现 vs. OpenEvolve 源码

| 概念 | 我们的实现 | OpenEvolve 源码 | 差异说明 |
|------|-----------|----------------|----------|
| 评估结果结构 | `EvaluationResult` | `evaluation_result.py:L10-62` | 几乎一致 |
| 阈值判定 | `passes_threshold()` | `evaluator.py:L668-707` | 逻辑一致 |
| 级联评估 | `CascadeEvaluator.evaluate()` | `evaluator.py:L360-535` | 我们用同步，源码用 async |
| 超时处理 | `time.time()` 检查 | `asyncio.wait_for()` | 源码更精确 |
| 工件存储 | `ArtifactStore` | `database.py` + `evaluator.py` | 源码集成在数据库中 |
| 阶段函数 | `sort_stage1/2/3` | `evaluate_stage1/2/3` | 命名惯例：`evaluate_stageN` |
| 重试机制 | 未实现 | `evaluator.py:L153-296` | 源码有 3 次重试 |
| 并行评估 | 未实现 | `TaskPool` (`async_utils.py`) | 下一章实现 |

### OpenEvolve 配置默认值

```yaml
evaluator:
  timeout: 300              # 每阶段超时（秒）
  max_retries: 3            # 失败重试次数
  cascade_evaluation: true  # 启用级联
  cascade_thresholds:       # 三阶段阈值
    - 0.5   # Stage 1: 基础验证
    - 0.75  # Stage 2: 中等测试
    - 0.9   # Stage 3: 全面评估
  parallel_evaluations: 1   # 并行评估数
```

### 关键源码位置

| 功能 | 文件 | 行号 |
|------|------|------|
| EvaluationResult 类 | `evaluation_result.py` | L10-62 |
| 级联评估主逻辑 | `evaluator.py` | L360-535 |
| 阈值判定 | `evaluator.py` | L668-707 |
| 超时处理 | `evaluator.py` | L252-265 |
| 工件收集 | `evaluator.py` | L150-242 |
| 评估配置 | `config.py` | L356-385 |
| 级联验证测试 | `test_cascade_validation.py` | L38-245 |

## 13. 本章总结

```mermaid
flowchart TB
    subgraph "级联评估系统"
        P[程序] --> S1[Stage 1<br/>基础验证]
        S1 -->|通过 ≥0.5| S2[Stage 2<br/>正确性测试]
        S1 -->|未通过| F1[淘汰 + 收集 artifacts]
        S2 -->|通过 ≥0.75| S3[Stage 3<br/>全面评估]
        S2 -->|未通过| F2[淘汰 + 收集 artifacts]
        S3 --> R[最终结果]
    end
    
    subgraph "工件系统"
        F1 --> AS[ArtifactStore]
        F2 --> AS
        R --> AS
        AS -->|< 32KB| MEM[内存存储]
        AS -->|≥ 32KB| DISK[磁盘存储]
    end
    
    subgraph "反馈循环"
        AS --> FB[格式化失败信息]
        FB --> LLM[LLM 变异器]
        LLM --> P
    end
```

### 核心收获

| 概念 | 一句话总结 |
|------|----------|
| 级联评估 | 逐步增加评估强度，早淘汰 = 省时间 |
| 阈值判定 | 有 combined_score 就用，没有就取平均 |
| 工件系统 | 评估的副产品（日志、错误信息）也是有用的数据 |
| 反馈循环 | 失败信息喂给 LLM，让变异更有针对性 |

### 下一章预告

级联评估解决了**评估效率**的问题，但 LLM 每次只看到**完整的代码**来做变异。  
当代码很长时（几百行甚至上千行），让 LLM 重写全部代码既慢又浪费。  
下一章我们实现 **基于 Diff 的进化** —— LLM 只输出改动部分，像 `git diff` 一样精准修改。

## 14. 保存模块

将本章实现的级联评估器保存到 `our-implementation/` 目录。

In [ ]:
save_code = '''
"""级联评估器 - 第五章实现

多阶段渐进式评估，提前淘汰低质量程序。
"""
from dataclasses import dataclass, field
from typing import Dict, Union, List, Callable, Optional
import time
import os
import json


@dataclass
class EvaluationResult:
    """评估结果：指标 + 工件"""
    metrics: Dict[str, float]
    artifacts: Dict[str, Union[str, bytes]] = field(default_factory=dict)
    
    def has_artifacts(self) -> bool:
        return len(self.artifacts) > 0
    
    def get_total_artifact_size(self) -> int:
        total = 0
        for v in self.artifacts.values():
            if isinstance(v, bytes):
                total += len(v)
            elif isinstance(v, str):
                total += len(v.encode("utf-8"))
            else:
                total += len(str(v).encode("utf-8"))
        return total


def passes_threshold(metrics: dict, threshold: float) -> bool:
    """判断指标是否达到阈值"""
    if "combined_score" in metrics:
        return float(metrics["combined_score"]) >= threshold
    valid = [
        float(v) for k, v in metrics.items()
        if k != "error" and isinstance(v, (int, float))
    ]
    if not valid:
        return False
    return sum(valid) / len(valid) >= threshold


class CascadeEvaluator:
    """级联评估器"""
    
    def __init__(self, stages, thresholds=None, timeout=30.0):
        self.stages = stages
        self.thresholds = thresholds or [0.5, 0.75, 0.9]
        self.timeout = timeout
    
    def evaluate(self, program_code: str) -> EvaluationResult:
        merged_metrics = {}
        merged_artifacts = {}
        start_time = time.time()
        
        for i, (stage_fn, threshold) in enumerate(zip(self.stages, self.thresholds)):
            stage_name = f"stage{i+1}"
            if time.time() - start_time >= self.timeout:
                merged_metrics[f"{stage_name}_passed"] = 0.0
                merged_artifacts["failure_reason"] = "timeout"
                break
            try:
                result = stage_fn(program_code)
                if isinstance(result, EvaluationResult):
                    stage_metrics = result.metrics
                    stage_artifacts = result.artifacts
                elif isinstance(result, dict):
                    stage_metrics = result
                    stage_artifacts = {}
                else:
                    stage_metrics = {}
                    stage_artifacts = {}
                merged_metrics.update(stage_metrics)
                merged_artifacts.update(stage_artifacts)
                if not passes_threshold(stage_metrics, threshold):
                    merged_metrics[f"{stage_name}_passed"] = 0.0
                    merged_artifacts["failure_stage"] = stage_name
                    break
                else:
                    merged_metrics[f"{stage_name}_passed"] = 1.0
            except Exception as e:
                merged_metrics[f"{stage_name}_passed"] = 0.0
                merged_artifacts["failure_stage"] = stage_name
                merged_artifacts["exception"] = str(e)
                break
        
        return EvaluationResult(metrics=merged_metrics, artifacts=merged_artifacts)


class ArtifactStore:
    """工件存储器"""
    THRESHOLD = 32 * 1024
    
    def __init__(self, disk_dir="/tmp/artifacts"):
        self.memory_store = {}
        self.disk_dir = disk_dir
        os.makedirs(disk_dir, exist_ok=True)
    
    def store(self, program_id, result):
        if not result.has_artifacts():
            return
        if result.get_total_artifact_size() < self.THRESHOLD:
            self.memory_store[program_id] = result.artifacts
        else:
            path = os.path.join(self.disk_dir, f"{program_id}.json")
            with open(path, "w") as f:
                json.dump({k: str(v) for k, v in result.artifacts.items()}, f)
    
    def retrieve(self, program_id):
        if program_id in self.memory_store:
            return self.memory_store[program_id]
        path = os.path.join(self.disk_dir, f"{program_id}.json")
        if os.path.exists(path):
            with open(path) as f:
                return json.load(f)
        return {}
'''

import os
os.makedirs('../our-implementation', exist_ok=True)
with open('../our-implementation/cascade_evaluator.py', 'w', encoding='utf-8') as f:
    f.write(save_code)

print('已保存到 our-implementation/cascade_evaluator.py ✓')